In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_groq import ChatGroq

load_dotenv()

embedding_model=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

persist_directory="db/chroma_db"

db=Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

model=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

chat_history=[]

def ask_questions(query):

    if chat_history:

        message=[
        SystemMessage(content="Given the chat history, rewrite the new question to be standalone and searchable. Just return the rewritten question")
        ]+chat_history+[
        HumanMessage(content=f"New question :{query}")]

        response=model.invoke(message)
        user_question=response.content.strip()

    else:
        user_question=query

    retriever=db.as_retriever(search_kwargs={"k":3})
    retriever_result=retriever.invoke(user_question)

    combined_input=f"""Based on the following documents, please answer this question:{query}
    Documents:
    {chr(10).join([f"-{doc.page_content}" for doc in retriever_result])}
    Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
    """""

    message=[SystemMessage(content="You are a helpful assistant."),
            HumanMessage(content=combined_input)]

    result=model.invoke(message)
    answer=result.content

    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=answer))

    print("Answer:")
    print(answer)

    return answer

if __name__ == "__main__":
    while True:
        query=input("Enter the question (q to quit):")

        if query.lower()=="q":
            break
        ask_questions(query)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Answer:
The two main components of the Transformer architecture are:

1. **Encoder**: Consists of N layers, each with two sub-layers: multi-head attention and feed forward layers, with add & norm operations.
2. **Decoder**: Also consists of N layers, with an additional sub-layer for masked multi-head attention, and includes add & norm operations.

These components work together to enable the Transformer model to handle input sequences and generate output sequences. The encoder takes in a sequence of tokens and outputs a sequence of vectors, which are then used by the decoder to generate an output sequence, one element at a time. 

In more detail, the Transformer model consists of an encoder and a decoder, both using stacked self-attention and point-wise, fully connected layers. The Transformer architecture includes multi-head attention, masked multi-head attention, feed forward layers, and positional encoding. 

References to the encoder-decoder structure can be found in literature [5,